# Fabric Workspace Inventory

Extracts comprehensive object-level information for all Microsoft Fabric workspaces under the target capacities using **Admin REST APIs**.

**Notebook structure (modular cells):**

| Cell | Purpose |
|------|---------|
| 2 | Imports |
| 3 | Configuration — target capacity names |
| 4 | `admin_api_get()` — generic Admin API caller with pagination & rate-limit retry |
| 5 | `resolve_capacities()` — map capacity names → IDs |
| 6 | `fetch_workspaces()` — retrieve workspaces via Admin API with `$expand` |
| 7 | `format_permissions()` — format user permission strings |
| 8 | `get_datasources()` / `format_connection_string()` — extract connection details |
| 9 | `get_refresh_schedule()` / `get_refresh_history()` — refresh metadata |
| 10 | `process_semantic_models()` / `process_dataflows()` / `process_reports()` — per-item-type processors |
| 11 | `fetch_user_activity()` — Admin activity events with multi-day support |
| 12 | `get_fabric_inventory()` — orchestrator that ties everything together |
| 13 | Execute & display results |

**Prerequisites:** Fabric Administrator role + tenant settings (see README).

In [ ]:
import sempy.fabric as fabric
import pandas as pd
import time
from datetime import datetime, timedelta
from typing import Optional

In [ ]:
# 1. Configuration - Define your specific capacities
target_capacity_names = [
    "lakfabuswest2"
]

In [ ]:
# ── Admin API helper with pagination & retry ──

MAX_RETRIES = 3
DEFAULT_RETRY_SECONDS = 5


def admin_api_get(client, url: str, paginated: bool = False):
    """
    Call a Fabric Admin REST API endpoint.

    Args:
        client:    PowerBIRestClient instance.
        paginated: If True, follows continuationUri / @odata.nextLink and
                   collects all pages into a single list.

    Returns:
        dict (single-page) or list[dict] (paginated).
    """
    results = []
    retries = 0

    while url:
        response = client.get(url)

        # ── rate-limit handling ──
        if response.status_code == 429:
            retries += 1
            if retries > MAX_RETRIES:
                raise Exception(f"Rate-limited {MAX_RETRIES} times on {url}")
            wait = int(response.headers.get("Retry-After", DEFAULT_RETRY_SECONDS))
            print(f"    Rate limited – retrying in {wait}s (attempt {retries}/{MAX_RETRIES})…")
            time.sleep(wait)
            continue

        if response.status_code != 200:
            raise Exception(f"{response.status_code} {response.reason} for url: {url}")

        data = response.json()
        retries = 0  # reset on success

        if paginated:
            results.extend(data.get("value", []))
            url = data.get("continuationUri") or data.get("@odata.nextLink")
        else:
            return data

    return results

In [ ]:
# ── Resolve target capacities ──

def resolve_capacities(capacity_names: list[str]) -> tuple[dict, set]:
    """
    Map capacity display names to IDs.

    Returns:
        id_to_name: dict mapping lowercase capacity-id → display name.
        target_ids: set of lowercase capacity-ids matching *capacity_names*.
    """
    caps_df = fabric.list_capacities()
    lower_targets = {n.lower() for n in capacity_names}

    id_to_name = {
        str(row["Id"]).lower(): row["Display Name"]
        for _, row in caps_df.iterrows()
    }
    target_ids = {
        cid for cid, name in id_to_name.items()
        if name.lower() in lower_targets
    }

    if not target_ids:
        available = list(id_to_name.values())
        raise ValueError(
            f"No capacities matched {capacity_names}. Available: {available}"
        )

    print(f"  Matched {len(target_ids)} capacity ID(s) for {capacity_names}")
    return id_to_name, target_ids

In [ ]:
# ── Fetch workspaces via Admin API ──

def fetch_workspaces(client, target_ids: set[str]) -> list[dict]:
    """
    Fetch all tenant workspaces using the Admin API with expanded metadata,
    then filter to those belonging to *target_ids* capacities.
    """
    print("  Fetching workspaces (Admin API with $expand)…")
    all_ws = admin_api_get(
        client,
        "/v1.0/myorg/admin/groups?$top=5000&$expand=users,datasets,dataflows,reports",
        paginated=True,
    )
    filtered = [
        ws for ws in all_ws
        if str(ws.get("capacityId", "")).lower() in target_ids
    ]
    print(f"  {len(filtered)} workspaces matched target capacities.")
    if not filtered:
        raise ValueError(
            "No workspaces found for the target capacities. "
            "Verify capacity names and admin permissions."
        )
    return filtered

In [ ]:
# ── Extract user permissions ──

def format_permissions(users: list[dict]) -> str:
    """Format the expanded *users* list from the Admin groups endpoint."""
    if not users:
        return "No users found"

    entries = []
    for u in users:
        identifier = (
            u.get("emailAddress")
            or u.get("displayName")
            or u.get("identifier", "Unknown")
        )
        role = u.get("groupUserAccessRight", u.get("principalType", "Unknown"))
        p_type = u.get("principalType", "")
        entries.append(f"{identifier} ({role}, {p_type})")
    return " | ".join(entries)

In [ ]:
# ── Extract datasource connection details ──

def format_connection_string(source: dict) -> str:
    """Build a human-readable connection string from a datasource dict."""
    details = source.get("connectionDetails") or {}
    ds_type = source.get("datasourceType", "Unknown")
    server = details.get("server", "")
    database = details.get("database", "")
    if server:
        return f"{ds_type}: {server}/{database}"
    path = details.get("path", "")
    url = details.get("url", "")
    return f"{ds_type}: {path or url or 'N/A'}"


def get_datasources(client, admin_path: str, item_name: str, errors: list) -> str:
    """
    Fetch datasources for a semantic model or dataflow.

    Args:
        admin_path: e.g. "/v1.0/myorg/admin/datasets/{id}/datasources"
        item_name:  used for error logging.
        errors:     mutable list to append error dicts to.
    """
    try:
        resp = admin_api_get(client, admin_path)
        sources = resp.get("value", []) if isinstance(resp, dict) else resp
        if sources:
            return "; ".join(format_connection_string(s) for s in sources)
        return "No datasources"
    except Exception as e:
        err_str = str(e)
        # 401 is expected for system-generated, push, streaming, or Direct Lake datasets
        if "401" in err_str:
            return "N/A (system-generated or unsupported model type)"
        errors.append({"Item": item_name, "Type": "Datasource", "Error": err_str[:200]})
        return f"Error: {err_str[:100]}"        

In [ ]:
# ── Extract refresh schedule & history ──

def get_refresh_schedule(client, dataset_id: str) -> str:
    """Return a formatted refresh-schedule string for a semantic model."""
    try:
        sched = admin_api_get(client, f"/v1.0/myorg/admin/datasets/{dataset_id}/refreshSchedule")
        if not sched:
            return "N/A"
        if not sched.get("enabled"):
            return "Disabled"
        freq = sched.get("frequency", "Unknown")
        times = ", ".join(sched.get("times", []))
        tz = sched.get("localTimeZoneId", "")
        return f"Every {freq} at [{times}] ({tz})"
    except Exception as e:
        if "401" in str(e) or "404" in str(e):
            return "N/A (not refreshable)"
        return f"Error: {str(e)[:80]}"


def get_refresh_history(client, dataset_id: str, top: int = 3, errors: list | None = None) -> str:
    """Return the last *top* refresh runs as a formatted string."""
    try:
        resp = admin_api_get(client, f"/v1.0/myorg/admin/datasets/{dataset_id}/refreshes?$top={top}")
        runs = resp.get("value", []) if isinstance(resp, dict) else resp
        if not runs:
            return "No history"
        entries = []
        for h in runs[:top]:
            status = h.get("status", "Unknown")
            start = h.get("startTime", "")[:19]
            end = h.get("endTime", "")[:19]
            entries.append(f"{status} ({start} → {end})")
        return " | ".join(entries)
    except Exception as e:
        if "401" in str(e) or "404" in str(e):
            return "N/A (not refreshable)"
        if errors is not None:
            errors.append({"Item": dataset_id, "Type": "RefreshHistory", "Error": str(e)[:200]})
        return "N/A"

In [ ]:
# ── Process workspace items into inventory rows ──

def _base_row(cap_name, ws_name, ws_id, perm_details) -> dict:
    """Shared columns for every inventory row."""
    return {
        "CapacityName": cap_name,
        "WorkspaceName": ws_name,
        "WorkspaceId": ws_id,
        "UserPermissions": perm_details,
    }


def process_semantic_models(client, datasets: list[dict], base: dict, errors: list) -> list[dict]:
    """Extract inventory rows for all semantic models in a workspace."""
    rows = []
    for ds in datasets:
        ds_id = ds.get("id", "N/A")
        ds_name = ds.get("name", "Unnamed")

        conn = get_datasources(client, f"/v1.0/myorg/admin/datasets/{ds_id}/datasources", ds_name, errors)
        sched = get_refresh_schedule(client, ds_id)
        hist = get_refresh_history(client, ds_id, errors=errors)

        rows.append({
            **base,
            "ItemName": ds_name,
            "ItemType": "SemanticModel",
            "ItemId": ds_id,
            "ConnectionDetails": conn,
            "RefreshSchedule": sched,
            "RefreshHistory": hist,
            "StorageMode": ds.get("targetStorageMode", "N/A"),
            "ConfiguredBy": ds.get("configuredBy", "N/A"),
        })
    return rows


def process_dataflows(client, dataflows: list[dict], base: dict, errors: list) -> list[dict]:
    """Extract inventory rows for all dataflows in a workspace."""
    rows = []
    for df in dataflows:
        df_id = df.get("objectId", df.get("id", "N/A"))
        df_name = df.get("name", "Unnamed")

        conn = get_datasources(client, f"/v1.0/myorg/admin/dataflows/{df_id}/datasources", df_name, errors)

        rows.append({
            **base,
            "ItemName": df_name,
            "ItemType": "Dataflow",
            "ItemId": df_id,
            "ConnectionDetails": conn,
            "RefreshSchedule": "N/A",
            "RefreshHistory": "N/A",
            "StorageMode": "N/A",
            "ConfiguredBy": df.get("configuredBy", "N/A"),
        })
    return rows


def process_reports(reports: list[dict], base: dict) -> list[dict]:
    """Extract inventory rows for all reports in a workspace."""
    return [
        {
            **base,
            "ItemName": rpt.get("name", "Unnamed"),
            "ItemType": "Report",
            "ItemId": rpt.get("id", "N/A"),
            "ConnectionDetails": f"Bound to dataset: {rpt.get('datasetId', 'N/A')}",
            "RefreshSchedule": "N/A (Report)",
            "RefreshHistory": "N/A (Report)",
            "StorageMode": "N/A",
            "ConfiguredBy": rpt.get("createdBy", "N/A"),
        }
        for rpt in reports
    ]

In [ ]:
# ── Fetch user activity events ──

def fetch_user_activity(
    client,
    workspace_ids: set[str],
    lookback_days: int = 1,
    errors: list | None = None,
) -> list[dict]:
    """
    Fetch user activity events from the Admin API for a 1-day window
    and filter to the given workspace IDs.

    The Admin activity API only supports 1-day windows, so we iterate
    over the last *lookback_days* days.
    """
    activity_rows = []
    now = datetime.utcnow().replace(hour=0, minute=0, second=0)

    for day_offset in range(lookback_days):
        # Activity API requires start & end within the SAME UTC day
        query_date = now - timedelta(days=day_offset + 1)
        start_str = query_date.strftime("%Y-%m-%dT00:00:00")
        end_str = query_date.strftime("%Y-%m-%dT23:59:59")

        try:
            events = admin_api_get(
                client,
                f"/v1.0/myorg/admin/activityevents?startDateTime=%27{start_str}%27&endDateTime=%27{end_str}%27",
                paginated=True,
            )
            for act in events:
                if act.get("WorkspaceId") in workspace_ids:
                    activity_rows.append({
                        "WorkspaceName": act.get("WorkspaceName", ""),
                        "Operation": act.get("Operation", ""),
                        "User": act.get("UserId", ""),
                        "ItemName": act.get("ArtifactName", act.get("DatasetName", "")),
                        "Timestamp": act.get("CreationTime", ""),
                    })
        except Exception as e:
            msg = f"Activity day {query_date.date()}: {e}"
            print(f"    ⚠ {msg}")
            if errors is not None:
                errors.append({"Item": "ActivityEvents", "Type": "UserActivity", "Error": str(msg)[:200]})

    print(f"  {len(activity_rows)} activity events captured ({lookback_days}-day window).")
    return activity_rows

In [ ]:
# ── Orchestrator: run full inventory extraction ──

def get_fabric_inventory(
    capacity_names: list[str],
    workspace_names: list[str] | None = None,
    activity_lookback_days: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    End-to-end inventory extraction using Power BI Admin APIs.

    Args:
        capacity_names:          List of capacity display names to scan.
        workspace_names:         Optional list of workspace names to limit the scan.
                                 If None, all workspaces in the target capacities are scanned.
        activity_lookback_days:  Number of past days of user-activity to fetch (max 30).

    Returns:
        (inventory_df, activity_df, errors_df)
    """
    client = fabric.PowerBIRestClient()
    errors: list[dict] = []

    # 1. Capacities
    print("Step 1: Resolving capacities…")
    id_to_name, target_ids = resolve_capacities(capacity_names)

    # 2. Workspaces
    print("Step 2: Fetching workspaces…")
    workspaces = fetch_workspaces(client, target_ids)

    # 2b. Filter to specific workspaces if requested
    if workspace_names:
        lower_ws = {n.lower() for n in workspace_names}
        workspaces = [ws for ws in workspaces if ws.get("name", "").lower() in lower_ws]
        if not workspaces:
            raise ValueError(
                f"No workspaces matched {workspace_names} in the target capacities."
            )
        print(f"  Filtered to {len(workspaces)} workspace(s): {workspace_names}")

    # 3. Inventory
    print("Step 3: Building item inventory…")
    inventory_rows: list[dict] = []

    for ws in workspaces:
        ws_id = ws["id"]
        ws_name = ws.get("name", "Unnamed")
        cap_name = id_to_name.get(str(ws.get("capacityId", "")).lower(), "Unknown")
        print(f"    {ws_name}…")

        perm = format_permissions(ws.get("users", []))
        base = _base_row(cap_name, ws_name, ws_id, perm)

        inventory_rows.extend(process_semantic_models(client, ws.get("datasets", []), base, errors))
        inventory_rows.extend(process_dataflows(client, ws.get("dataflows", []), base, errors))
        inventory_rows.extend(process_reports(ws.get("reports", []), base))

    # 4. Activity
    print("Step 4: Fetching user activity…")
    ws_ids = {ws["id"] for ws in workspaces}
    activity_rows = fetch_user_activity(client, ws_ids, activity_lookback_days, errors)

    return (
        pd.DataFrame(inventory_rows),
        pd.DataFrame(activity_rows),
        pd.DataFrame(errors),
    )

In [ ]:
# ── Execute extraction ──

# For dev/testing: pass workspace_names to scan a single workspace quickly
# For full run:   set workspace_names=None (or remove the parameter)

inventory_df, activity_df, errors_df = get_fabric_inventory(
    capacity_names=target_capacity_names,
    workspace_names=["MyTestWorkspace"],  # ← change or set to None for all workspaces
    activity_lookback_days=1,
)

print(f"\n{'=' * 60}")
print("INVENTORY SUMMARY")
print(f"{'=' * 60}")
print(f"Total items: {len(inventory_df)}")

if not inventory_df.empty:
    print(f"\nBy type:\n{inventory_df['ItemType'].value_counts().to_string()}")
    print(f"\nBy capacity:\n{inventory_df['CapacityName'].value_counts().to_string()}")

print(f"\nActivity events: {len(activity_df)}")

if not errors_df.empty:
    print(f"\n⚠ {len(errors_df)} errors encountered:")
    display(errors_df)

print(f"\n{'=' * 60}")
print("FULL INVENTORY:")
display(inventory_df)

if not activity_df.empty:
    print("\nUSER ACTIVITY:")
    display(activity_df)